# **Thames Valley Crime Data Analysis**
## Modelling: Predicting Crime Counts
---

### Objectives

- Answer Key Question 5: can we predict the number of crimes for an area, crime type and month?
- Compare a baseline with models suitable for the data that we have, then decide which is the best.
- Test the chosen model on data that it has not seen.
- Explain any limitations.

### Planned Steps

1. Load, check and prepare the cleaned data
2. Split the data into training and test data
3. Find the baseline
4. Compare the models
5. Test the chosen model
6. Conclusions

### Inputs

`../data/clean/crime_summary.csv` - Crime counts per month, season, area and crime type.

The summary file is used rather than `crime_clean.csv`, because the model predicts **how many** crimes there are. It is much smaller (0.3 MB), so it is quicker to load.

### Key Questions explored in this notebook

5. Can monthly crime counts be predicted for an area and crime type, and if so, which model does it best?

### Outputs

- A model comparison and test scores.
- A chart showing actual and predicted crime counts.
- Answer to Key Question 5.

### Additional Comments

- Cells in this notebook run top to bottom.
- The model predicts **counts**, not **rates**. Larger areas naturally have more crime (see Limitations in the README).

---
### Setup

Import required libraries:

- `matplotlib` - to draw the chart.
- `numpy` - for numerical calculations.
- `pandas` - to load the data and show results.
- `scikit-learn` - to prepare the data, to build, test and score the models.

Note: `%matplotlib inline` is used so charts display inside the notebook (see Bugs in README).

In [3]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Ensures that charts are displayed correctly in the notebook
%matplotlib inline

print("Libraries imported successfully")

Libraries imported successfully


---
### 1. Load, check and prepare the cleaned data

Here we will load and check the summary data to make sure nothing was lost or changed since we saved `crime_summary.csv` at the end of `01_ETL.ipynb`, and that it is ready for modelling.

We predict ***Count*** using ***Area***, ***Crime type*** and the **month**.

We will use **supervised learning**, as the data has a target (***Count***) for the model to learn from. It is a regression problem, because the target is a number.

We will prepare our data using **cyclical encoding**, so the models see the months as a continuous loop.

The year is not used, as from answering Question 3 we already know that crime only fell 0.307% (595 fewer crimes) from Year 1 to Year 3 - so it effectively stayed level.

In [9]:
# Load the data
summary = pd.read_csv("../data/clean/crime_summary.csv",
                      parse_dates=["Month"])

# print the rows and columns and check their type
print("Rows and columns:",summary.shape)
print("Total crimes: ", summary["Count"].sum())
print("Missing Values", summary.isnull().sum().sum())
summary.info()

summary.head()

Rows and columns: (6552, 5)
Total crimes:  579772
Missing Values 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6552 entries, 0 to 6551
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Month       6552 non-null   datetime64[ns]
 1   Season      6552 non-null   object        
 2   Area        6552 non-null   object        
 3   Crime type  6552 non-null   object        
 4   Count       6552 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 256.1+ KB


,Month,Season,Area,Crime type,Count
0,2023-10-01,Autumn,Bracknell Forest,Anti-social behaviour,81
1,2023-10-01,Autumn,Buckinghamshire,Anti-social behaviour,306
2,2023-10-01,Autumn,Cherwell,Anti-social behaviour,90
3,2023-10-01,Autumn,Milton Keynes,Anti-social behaviour,171
4,2023-10-01,Autumn,Oxford,Anti-social behaviour,125


We can see from the above that:
- the data has 6,552 rows, 5 columns, a total of 579,772 crimes and no missing values, all matching `01_ETL.ipynb`.
- ***Month*** was loaded as a date.
- nothing was lost or changed, so the data is ready to be prepared for modelling.

Next, we place the month into a loop using cyclical encoding.

In [13]:
# Place each month into a loop
month_number = summary["Month"].dt.month
summary["month_sin"] = np.sin(2 * np.pi * month_number / 12)
summary["month_cos"] = np.cos(2 * np.pi * month_number / 12)

# Show one row per month to check cyclical encoding
months = summary.drop_duplicates(["Month"]).head(36)
months[["Month", "month_sin", "month_cos"]].round(2)

,Month,month_sin,month_cos
0,2023-10-01,-0.87,0.50
13,2023-11-01,-0.50,0.87
26,2023-12-01,-0.00,1.00
39,2024-01-01,0.50,0.87
52,2024-02-01,0.87,0.50
65,2024-03-01,1.00,0.00
78,2024-04-01,0.87,-0.50
91,2024-05-01,0.50,-0.87
104,2024-06-01,0.00,-1.00
117,2024-07-01,-0.50,-0.87


Each month now has its own pair of values (sine (sin) and cosine (cos)). As we have asked for the full 36 months of data we can now see that the same values repeat each year, so the model sees each month as the same every year.

Because crime stayed the same level each year (only 0.307% lower, **Question 3**), leaving out the year makes little difference. It is noted as a limitation in the conclusions.